# nb_silver_to_gold
Silver Delta テーブルから Agent 2/3 向け Gold テーブルを生成します。


## 出力
`kpi.monthly_revenue`、`kpi.monthly_cost_detail`、`kpi.customer_count`、`kpi.fx_sensitivity`、`mobile_ai.risk_summary`、`ecommerce_ai.risk_summary`、`fintech_ai.risk_summary` を冪等に作成します。


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
WORKSPACE_NAME = "fabric_seworkshop_ws1"
SILVER_LAKEHOUSE_NAME = "lh_nexus6_silver"

def silver_table_path(name):
    schema, table = name.split(".", 1)
    return f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com/{SILVER_LAKEHOUSE_NAME}.Lakehouse/Tables/{schema}/{table}"
for schema in ["kpi", "mobile_ai", "ecommerce_ai", "fintech_ai"]:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema}")

def t(name):
    return spark.read.format("delta").load(silver_table_path(name))

def write_table(df, table):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)

def metric(df, year_col, name, value_col, unit, desc):
    return df.select(F.col(year_col).alias("year_month"), F.lit(name).alias("metric_name"), F.col(value_col).cast(DecimalType(16,4)).alias("metric_value"), F.lit(unit).alias("metric_unit"), F.lit(desc).alias("description"))

mobile_rev = t("mobile.usage_billing").groupBy("year_month").agg(F.sum("monthly_charge_jpy").alias("gross_revenue_jpy"))
mobile_cost = t("mobile.device_costs").groupBy("year_month").agg(F.sum("unit_cost_jpy").alias("total_cost_jpy"), F.sum(F.when(F.col("currency") == "USD", F.col("unit_cost")).otherwise(0)).alias("fx_exposure_usd"))
mobile_cust = t("mobile.contracts").groupBy("year_month").agg(F.countDistinct("customer_id").alias("active_customer_count"), F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("churned_customer_count"))
mobile_monthly = (mobile_rev.join(mobile_cost, "year_month", "full").join(mobile_cust, "year_month", "full")
    .fillna(0).withColumn("division", F.lit("mobile")))

ec_monthly = (t("ecommerce.orders").groupBy("year_month").agg(F.sum("order_amount_jpy").alias("gross_revenue_jpy"), F.sum("cost_price_jpy").alias("total_cost_jpy"), F.countDistinct("member_id").alias("active_customer_count"), F.sum(F.when(F.col("procurement_currency") == "USD", F.col("cost_price_jpy") / F.col("fx_rate_used")).otherwise(0)).alias("fx_exposure_usd"))
    .withColumn("division", F.lit("ecommerce")).withColumn("churned_customer_count", F.lit(0)))

fin_rev = t("fintech.revenue_risks").agg((F.sum("fee_revenue") + F.sum("interest_revenue") + F.sum("fx_revenue")).alias("gross_revenue_jpy"), F.sum("risk_loss").alias("total_cost_jpy")).withColumn("year_month", F.lit("2026-06"))
fin_cust = t("fintech.accounts").agg(F.countDistinct("user_id").alias("active_customer_count")).withColumn("year_month", F.lit("2026-06"))
fin_monthly = fin_rev.join(fin_cust, "year_month", "full").fillna(0).withColumn("division", F.lit("fintech")).withColumn("churned_customer_count", F.lit(0)).withColumn("fx_exposure_usd", F.lit(0))

monthly = mobile_monthly.unionByName(ec_monthly, allowMissingColumns=True).unionByName(fin_monthly, allowMissingColumns=True)
monthly = (monthly.withColumn("gross_margin_jpy", F.col("gross_revenue_jpy") - F.col("total_cost_jpy"))
    .withColumn("gross_margin_rate", F.when(F.col("gross_revenue_jpy") != 0, F.col("gross_margin_jpy") / F.col("gross_revenue_jpy")).otherwise(0))
    .withColumn("fx_exposure_other_jpy", F.lit(0).cast(DecimalType(14,2)))
    .select("year_month", "division", "gross_revenue_jpy", "total_cost_jpy", "gross_margin_jpy", "gross_margin_rate", "fx_exposure_usd", "fx_exposure_other_jpy", "active_customer_count", "churned_customer_count"))
write_table(monthly, "kpi.monthly_revenue")

cost_detail = (mobile_cost.select("year_month", F.lit("mobile").alias("division"), F.lit("device_cost").alias("cost_type"), F.col("total_cost_jpy"))
    .unionByName(t("ecommerce.orders").groupBy("year_month").agg(F.sum("cost_price_jpy").alias("total_cost_jpy")).select("year_month", F.lit("ecommerce").alias("division"), F.lit("cost_of_goods").alias("cost_type"), "total_cost_jpy"))
    .unionByName(fin_rev.select("year_month", F.lit("fintech").alias("division"), F.lit("risk_loss").alias("cost_type"), F.col("total_cost_jpy"))))
write_table(cost_detail, "kpi.monthly_cost_detail")
write_table(monthly.select("year_month", "division", "active_customer_count", "churned_customer_count"), "kpi.customer_count")
fx_sensitivity = (t("fintech.fx_rate_snapshots").groupBy("year_month", F.col("base_currency").alias("currency")).agg(F.avg("mid_rate").alias("avg_rate"))
    .join(monthly.groupBy("year_month").agg(F.sum("fx_exposure_usd").alias("exposure_amount")), "year_month", "left"))
write_table(fx_sensitivity, "kpi.fx_sensitivity")

mobile_metrics = metric(mobile_cost, "year_month", "overseas_procurement_cost", "total_cost_jpy", "JPY", "海外仕入を含む端末・設備コスト合計") \
    .unionByName(metric(t("mobile.contracts").groupBy("year_month").agg(F.sum("subsidy_amount").alias("v")), "year_month", "device_subsidy", "v", "JPY", "端末補助額合計")) \
    .unionByName(metric(t("mobile.mnp_history").filter(F.col("mnp_type") == "転出").groupBy("year_month").agg(F.count("*").alias("v")), "year_month", "mnp_out_count", "v", "件", "MNP転出件数")) \
    .unionByName(metric(t("mobile.installment_details").groupBy("year_month").agg(F.sum("monthly_payment_jpy").alias("v")), "year_month", "installment_payment", "v", "JPY", "分割払い月額合計")) \
    .unionByName(metric(t("mobile.crm_tickets").groupBy("year_month").agg(F.sum(F.when(F.col("cancel_flag"),1).otherwise(0)).alias("v")), "year_month", "cancel_ticket_count", "v", "件", "解約問い合わせ件数"))
write_table(mobile_metrics, "mobile_ai.risk_summary")

ec_metrics = metric(t("ecommerce.orders").groupBy("year_month").agg(F.sum("cost_price_jpy").alias("v")), "year_month", "cross_border_cost", "v", "JPY", "越境EC仕入コスト") \
    .unionByName(metric(t("ecommerce.orders").groupBy("year_month").agg((F.sum("gross_margin_jpy")/F.sum("order_amount_jpy")).alias("v")), "year_month", "gross_margin_rate", "v", "%", "粗利率")) \
    .unionByName(metric(t("ecommerce.point_events").groupBy("year_month").agg(F.sum("point_amount").alias("v")), "year_month", "point_cost", "v", "point", "ポイント還元コスト")) \
    .unionByName(metric(t("ecommerce.member_behaviors").filter(F.col("event_type") == "離脱").groupBy("year_month").agg(F.count("*").alias("v")), "year_month", "cart_abandon_count", "v", "件", "カート離脱件数")) \
    .unionByName(metric(t("ecommerce.campaign_reactions").groupBy("year_month").agg(F.count("*").alias("v")), "year_month", "campaign_reactions", "v", "件", "キャンペーン反応件数"))
write_table(ec_metrics, "ecommerce_ai.risk_summary")

fin_metrics = metric(t("fintech.fx_positions").groupBy("year_month").agg(F.sum("pnl_jpy").alias("v")), "year_month", "fx_position_pnl", "v", "JPY", "FXポジション損益合計") \
    .unionByName(metric(t("fintech.card_transactions").filter(F.col("overseas_flag")).groupBy("year_month").agg(F.sum("amount_jpy").alias("v")), "year_month", "overseas_card_amount", "v", "JPY", "海外カード決済額")) \
    .unionByName(metric(t("fintech.loan_balances").groupBy("year_month").agg(F.sum("principal_balance_jpy").alias("v")), "year_month", "loan_balance", "v", "JPY", "ローン残高合計")) \
    .unionByName(metric(t("fintech.loan_balances").groupBy("year_month").agg(F.avg("interest_rate").alias("v")), "year_month", "avg_interest_rate", "v", "%", "平均金利")) \
    .unionByName(metric(t("fintech.credit_reviews").filter(F.col("approval_status").isin("否認", "保留")).groupBy("year_month").agg(F.count("*").alias("v")), "year_month", "negative_credit_reviews", "v", "件", "否認・保留審査件数"))
write_table(fin_metrics, "fintech_ai.risk_summary")
